# 🧠 ChromaDB Fundamentals: Concept & Mechanics

Welcome to the first module of our ChromaDB & Local AI journey!  
The goal of this notebook is **not just to run code**, but to give you a deep, intuitive understanding of:
1. **Why Vector Databases exist** and how they differ from Relational Databases (SQL/PostgreSQL).
2. **The 4 core components** of ChromaDB: `Client`, `Collection`, `Embeddings`, and `Items`.
3. **How semantic search works**: Finding documents based on *meaning* rather than exact keyword matches.
4. **Metadata filtering**: Combining semantic intelligence with structured filters (e.g., filter by company, year, or department).

### 🔍 Conceptual Foundation: SQL vs. Vector Database

| Feature | Traditional Database (SQL/NoSQL) | Vector Database (ChromaDB) |
| :--- | :--- | :--- |
| **How it searches** | Exact matches (`WHERE company = 'Apple'`) or regex keywords | Semantic meaning / conceptual closeness |
| **Under the hood** | B-Trees, Hash Tables | High-Dimensional Vectors & HNSW Graphs |
| **Example Query** | Searching `"artificial intelligence profit"` only returns rows with those exact words. | Searching `"artificial intelligence profit"` matches `"TechNova reported $4.2B in cloud computing revenue driven by AI"` even if the words differ! |
| **Best for** | Transactions, inventory, structured accounts | Unstructured text, documents, knowledge bases, AI memory (RAG) |

--- 
## Step 1: Connecting to ChromaDB (In-Memory vs. Persistent)

ChromaDB provides two primary modes of operation:
- **Ephemeral Client (`chromadb.EphemeralClient()`)**: Runs in RAM only. When your Python process terminates, everything is deleted. Good for quick tests.
- **Persistent Client (`chromadb.PersistentClient(path=...)`)**: Writes documents, metadata (in SQLite), and vector indexes (HNSW binary graphs) directly to disk. When your script restarts, your data is still there!

Let's initialize a **Persistent Client** pointing to `./chroma_data`.

In [1]:
import chromadb
import os

# Define local directory where ChromaDB stores its database on disk
db_directory = os.path.join(os.getcwd(), "chroma_data")

# Initialize the persistent client
client = chromadb.PersistentClient(path=db_directory)

print(f" Connected to ChromaDB Persistent Client!")
print(f"📁 Data stored at: {db_directory}")

 Connected to ChromaDB Persistent Client!
📁 Data stored at: c:\codeX\learn-AI\implementing_chromaDB\chroma_data


--- 
## Step 2: What is a Collection?

In ChromaDB, a **Collection** is the equivalent of a **Table** in SQL or a **Collection** in MongoDB.  
All items inside a collection share:
1. The same vector space (same embedding dimensionality).
2. A distance metric (how similarity is calculated: `cosine`, `l2`, or `ip`).

> **Distance Metrics Explained**:
> - **Cosine Distance (`cosine`)** *(Default)*: Measures the angle between two vectors. Great for text because it ignores document length and focuses on direction/meaning. (Distance of `0.0` = identical meaning).
> - **L2 Euclidean Distance (`l2`)**: Measures straight-line geometric distance.
> - **Inner Product (`ip`)**: Dot product (useful for normalized embeddings).

We use `get_or_create_collection` so running this cell multiple times won't produce duplicate errors.

In [2]:
# Create or retrieve our collection
collection = client.get_or_create_collection(
    name="company_knowledge",
    metadata={"hnsw:space": "cosine"}  # Explicitly using Cosine Distance
)

print(f" Collection ready: '{collection.name}'")
print(f" Existing item count: {collection.count()}")

 Collection ready: 'company_knowledge'
 Existing item count: 0


--- 
## Step 3: Anatomical Breakdown of a Vector Record

When you save data into ChromaDB, each entry has 4 components:

1. **`id` (Required)**: A unique string identifier (e.g., `"doc_001"`, `"comp_meta_2024"`).
2. **`document` (Required if no embedding provided)**: The actual human-readable raw text chunk.
3. **`embedding` (Optional/Automatic)**: A list of floating-point numbers representing the text in vector space. If you don't provide one, ChromaDB automatically converts your `document` into an embedding using its default model (`all-MiniLM-L6-v2`, 384 dimensions)!
4. **`metadata` (Optional)**: A Python dictionary containing structured key-value pairs (e.g., `{"company": "Tesla", "year": 2024}`). This allows SQL-like filtering.

Let's add 5 realistic company records using `upsert()` (Update or Insert).

In [3]:
# 1. Raw Text Documents
documents = [
    "TechNova Inc. reported $4.2B in cloud computing revenue, driven by enterprise AI adoption in North America.",
    "BioHealth Labs received FDA approval for their innovative gene therapy targeting rare cardiovascular diseases.",
    "EcoEnergy Solutions installed 12 gigawatts of offshore wind capacity across the North Sea in fiscal year 2023.",
    "TechNova Inc. expanded its European cybersecurity operations with a new security operations center in Frankfurt.",
    "RetailPulse posted a 14% drop in physical store foot traffic as e-commerce sales reached 60% of total revenue."
]

# 2. Structured Metadata for each document
metadatas = [
    {"company": "TechNova", "industry": "Technology", "year": 2024, "topic": "cloud_revenue"},
    {"company": "BioHealth", "industry": "Healthcare", "year": 2024, "topic": "fda_approval"},
    {"company": "EcoEnergy", "industry": "CleanTech", "year": 2023, "topic": "wind_power"},
    {"company": "TechNova", "industry": "Technology", "year": 2024, "topic": "cybersecurity"},
    {"company": "RetailPulse", "industry": "Retail", "year": 2023, "topic": "store_sales"}
]

# 3. Unique IDs
ids = ["doc_001", "doc_002", "doc_003", "doc_004", "doc_005"]

# Upsert into the collection
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print(f" Stored {collection.count()} documents successfully!")

 Stored 5 documents successfully!


--- 
## Step 4: Semantic Querying (The Magic of Vector Search)

Here is what happens when you call `collection.query(query_texts=["..."])`:
1. ChromaDB passes your query text to the embedding model.
2. The model outputs a query vector.
3. ChromaDB's index compares that query vector against all vectors in the collection using **cosine distance**.
4. It returns the top `n_results` with the smallest distance.

> 💡 **Notice**: We will ask: `"Who is making money from artificial intelligence?"`  
> Even though neither "money" nor "artificial intelligence" appears verbatim in `"$4.2B in cloud revenue driven by AI"`, ChromaDB matches them because their **vector embeddings are close in meaning**!

In [7]:
query = "Who is making money from artificial intelligence?"

results = collection.query(
    query_texts=[query],
    n_results=2  # Return top 2 closest matches
)

print(f"🔎 Query: '{query}'\n")
for i in range(len(results["ids"][0])):
    doc_id = results["ids"][0][i]
    doc_text = results["documents"][0][i]
    meta = results["metadatas"][0][i]
    dist = results["distances"][0][i]
    
    print(f"--- Result #{i + 1} ---")
    print(f"  ID:       {doc_id}")
    print(f"  Distance: {dist:.4f} (Closer to 0 = more relevant)")
    print(f"  Company:  {meta['company']} [{meta['industry']}]")
    print(f"  Excerpt:  {doc_text}")

🔎 Query: 'Who is making money from artificial intelligence?'

--- Result #1 ---
  ID:       doc_001
  Distance: 0.4986 (Closer to 0 = more relevant)
  Company:  TechNova [Technology]
  Excerpt:  TechNova Inc. reported $4.2B in cloud computing revenue, driven by enterprise AI adoption in North America.
--- Result #2 ---
  ID:       doc_004
  Distance: 0.7506 (Closer to 0 = more relevant)
  Company:  TechNova [Technology]
  Excerpt:  TechNova Inc. expanded its European cybersecurity operations with a new security operations center in Frankfurt.


--- 
## Step 5: Metadata Filtering (Hybrid Search)

In enterprise datasets, semantic search alone isn't enough. You often need to say:  
- *"Search only inside TechNova's 2024 reports."*  
- *"Search only within CleanTech companies."*  

ChromaDB allows you to pass a `where` dictionary. The database filters by metadata first (or simultaneously with the vector scan) to ensure strict accuracy.

In [9]:
# Let's search for 'renewable green electrical power', but strictly filter for industry = 'CleanTech'
filtered_query = "renewable green electrical power"

filtered_results = collection.query(
    query_texts=[filtered_query],
    n_results=1,
    where={"industry": "CleanTech"}  # Metadata filter
)

print(f"🔎 Filtered Query: '{filtered_query}' (where industry = CleanTech)\n")
for i in range(len(filtered_results["ids"][0])):
    print(f"  Company:  {filtered_results['metadatas'][0][i]['company']}")
    print(f"  Content:  {filtered_results['documents'][0][i]}")
    print(f"  Distance: {filtered_results['distances'][0][i]:.4f}")

🔎 Filtered Query: 'renewable green electrical power' (where industry = CleanTech)

  Company:  EcoEnergy
  Content:  EcoEnergy Solutions installed 12 gigawatts of offshore wind capacity across the North Sea in fiscal year 2023.
  Distance: 0.5533


--- 
## Step 6: Peeking & Managing Records

You can also inspect records without running a vector query, or delete and update them as needed:

In [10]:
# 1. Peek at the first 2 items stored in the collection
print("👀 Collection Peek (First 2 items):")
print(collection.peek(limit=2))

# 2. Retrieve an exact document by ID
exact_item = collection.get(ids=["doc_002"])
print("\n🎯 Exact lookup for 'doc_002':")
print(f"Document: {exact_item['documents'][0]}")
print(f"Metadata: {exact_item['metadatas'][0]}")

👀 Collection Peek (First 2 items):
{'ids': ['doc_001', 'doc_002'], 'embeddings': array([[ 5.62516898e-02,  2.00774260e-02, -7.43023632e-03,
        -2.05483828e-02, -2.33951155e-02, -6.94103166e-03,
         1.66850593e-02,  5.59643377e-03,  3.77924703e-02,
         6.80382922e-02, -2.69279908e-02,  2.39414745e-03,
         9.16683376e-02, -4.25927043e-02, -2.23562662e-02,
         6.66869655e-02,  3.24177407e-02, -1.93343341e-01,
         5.80854230e-02,  7.21751573e-03,  2.00914219e-02,
         2.88320780e-02, -2.54620728e-03, -3.94112095e-02,
         8.85433927e-02,  5.02110980e-02,  1.11793624e-02,
        -2.53331307e-02, -3.03456578e-02, -2.30393168e-02,
        -7.98757225e-02,  4.67699468e-02,  2.47416925e-02,
         7.38995615e-03, -1.40895955e-02, -2.65899282e-02,
         2.98167113e-02,  1.10813417e-02,  4.29282710e-03,
        -4.14414797e-03,  4.31929491e-02, -8.85973349e-02,
        -7.15352148e-02, -1.88445356e-02,  4.70661819e-02,
         1.85704343e-02, -1.910982

--- 
## Summary & Next Step

In this notebook, we learned:
1. **`PersistentClient`** saves data to your local disk in `./chroma_data`.
2. **`Collection`** holds vectors, documents, and metadata using a distance metric (`cosine`).
3. **`upsert()`** safely inserts or updates documents.
4. **`query()`** calculates semantic distances between query text and documents.
5. **`where` filters** combine structured SQL-like filtering with semantic search.

**Next Up**: In Notebook 2, we will tackle **handling large company datasets** (chunking long documents, batching thousands of records, and designing scalable metadata schemas)!